# Project 03 — Cybersecurity Intrusion Detection

**Dataset:** UNSW-NB15 (Parquet mirror on Kaggle)  
**Methods:** Unsupervised Isolation Forest and supervised Random Forest  
**Goal:** Learn the difference between anomaly detection and attack classification, and diagnose false alarms.

This is a **learning benchmark**, not a deployable intrusion detection system. The test split is a benchmark evaluation, not a live-network prospective test.

**Reproduction:** Attach the Kaggle dataset `dhoogla/unswnb15` and run the notebook from top to bottom. No external downloads or secrets are required.

## 1. Load and inspect the data

The Parquet files are loaded from the Kaggle `dhoogla/unswnb15` input. The supplied test split remains unused until the final evaluation.

In [ ]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/dhoogla/unswnb15"

train = pd.read_parquet(
    f"{DATA_DIR}/UNSW_NB15_training-set.parquet"
)

test = pd.read_parquet(
    f"{DATA_DIR}/UNSW_NB15_testing-set.parquet"
)

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

print("\nColumn names:")
print(train.columns.tolist())

print("\nFirst five observations:")
display(train.head())

print("\nAttack labels:")
print(train["label"].value_counts())

## 2. Explore attack categories

The dataset is attack-heavy: **119,341 attack** and **56,000 normal** training connections. Category representation is highly uneven.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count examples in each attack category
attack_counts = train["attack_cat"].value_counts()

print(attack_counts)

plt.figure(figsize=(10, 5))
attack_counts.sort_values().plot(kind="barh")

plt.title("UNSW-NB15: Network Traffic Categories")
plt.xlabel("Number of connections")
plt.ylabel("Traffic category")
plt.tight_layout()
plt.show()

## 3. Baseline: Isolation Forest with five features

Fit **only on normal training traffic**. Hold out 20% of the original training split for development validation. Labels are used for evaluation, not fitting this anomaly detector.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

features = ["dur", "spkts", "dpkts", "sbytes", "dbytes"]

# Separate normal traffic and attacks
normal = train[train["label"] == 0]
attacks = train[train["label"] == 1]

# Hold out some normal connections
normal_train, normal_val = train_test_split(
    normal, test_size=0.2, random_state=42
)

# Hold out some attacks for evaluation
_, attack_val = train_test_split(
    attacks, test_size=0.2, random_state=42
)

# Train using normal traffic ONLY
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(normal_train[features])

# Evaluate normal and attack connections
validation = pd.concat([normal_val, attack_val])

predictions = model.predict(validation[features])

# Isolation Forest outputs 1 = normal, -1 = anomaly
predicted_labels = (predictions == -1).astype(int)

print("Confusion matrix:")
print(confusion_matrix(validation["label"], predicted_labels))

print("\nClassification report:")
print(classification_report(
    validation["label"],
    predicted_labels,
    target_names=["Normal", "Attack"],
    zero_division=0
))

## 4. Anomaly threshold sensitivity

Higher anomaly scores indicate more unusual connections. Percentiles are determined using **normal training observations only**; the values illustrate the precision–recall/false-alarm trade-off.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# Higher scores mean more anomalous
scores = -model.score_samples(validation[features])

actual = validation["label"].to_numpy()

results = []
normal_scores = -model.score_samples(normal_train[features])

for percentile in [80, 90, 95, 97, 99]:

    # Threshold based only on normal training data
    threshold = np.percentile(normal_scores, percentile)

    predicted = (scores >= threshold).astype(int)

    results.append({
        "Threshold_Percentile": percentile,
        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "Recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "False_Alarm_Rate": np.mean(
            predicted[actual == 0] == 1
        )
    })

display(pd.DataFrame(results).round(3))

## 5. Enhanced Isolation Forest with ten features

Keep the algorithm and split fixed; add traffic rates, loads, and recent connection statistics. This is an exploratory feature comparison, not an isolated causal attribution of which feature helps.

In [ ]:
enhanced_features = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes",
    "rate", "sload", "dload",
    "ct_src_dport_ltm", "ct_dst_sport_ltm"
]

enhanced_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

# Train only on normal connections
enhanced_model.fit(normal_train[enhanced_features])

# Predict on the same validation data
enhanced_pred = (
    enhanced_model.predict(validation[enhanced_features]) == -1
).astype(int)

from sklearn.metrics import precision_score, recall_score

print("Original model:")
print("Precision:", round(precision_score(
    validation["label"], predicted_labels), 3))
print("Recall:", round(recall_score(
    validation["label"], predicted_labels), 3))

print("\nEnhanced model:")
print("Precision:", round(precision_score(
    validation["label"], enhanced_pred), 3))
print("Recall:", round(recall_score(
    validation["label"], enhanced_pred), 3))

## 6. Attack-category subgroup analysis

An aggregate attack-recall value can hide significant disparities across attack categories. Small groups such as Worms have particularly uncertain estimates.

In [ ]:
attack_results = validation.copy()
attack_results["detected"] = enhanced_pred

detection_by_type = (
    attack_results[attack_results["label"] == 1]
    .groupby("attack_cat", observed=True)["detected"]
    .agg(["count", "mean"])
)

detection_by_type["Detection_Rate_%"] = (
    detection_by_type["mean"] * 100
)

display(
    detection_by_type[
        ["count", "Detection_Rate_%"]
    ].sort_values("Detection_Rate_%").round(2)
)

## 7. Recall excluding Generic attacks

This reveals whether aggregate recall is dominated by the common and easily detected Generic attack category.

In [ ]:
# Evaluate recall excluding Generic attacks

non_generic = attack_results[
    (attack_results["label"] == 1) &
    (attack_results["attack_cat"] != "Generic")
]

print("Non-Generic attack observations:", len(non_generic))

print(
    "Recall excluding Generic:",
    round(non_generic["detected"].mean(), 3)
)

print(
    "Overall attack recall:",
    round(
        attack_results.loc[
            attack_results["label"] == 1, "detected"
        ].mean(), 3
    )
)

## 8. Supervised Random Forest comparison

Train on labelled normal and attack examples from the original training portion and evaluate on the **same development validation** observations.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# Same normal training data as before, plus attack examples
attack_train = attacks.drop(index=attack_val.index)

supervised_train = pd.concat(
    [normal_train, attack_train]
)

# Use the same validation observations
supervised_val = pd.concat(
    [normal_val, attack_val]
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    supervised_train[enhanced_features],
    supervised_train["label"]
)

rf_pred = rf.predict(
    supervised_val[enhanced_features]
)

print("Supervised Random Forest:")
print(classification_report(
    supervised_val["label"],
    rf_pred,
    target_names=["Normal", "Attack"],
    zero_division=0
))

# Evaluate non-Generic attacks
mask = (
    (supervised_val["label"] == 1) &
    (supervised_val["attack_cat"] != "Generic")
)

print(
    "Non-Generic attack recall:",
    round(rf_pred[mask.to_numpy()].mean(), 3)
)

## 9. False alarms and missed attacks

Measure false positives separately from true-positive attack detection.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    supervised_val["label"],
    rf_pred
)

tn, fp, fn, tp = cm.ravel()

print("Normal correctly identified:", tn)
print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks detected:", tp)

print("\nFalse alarm rate:", round(fp / (fp + tn) * 100, 2), "%")
print("Attack detection rate:", round(tp / (tp + fn) * 100, 2), "%")

## 10. Independent test evaluation

Run this section **once after model development**. Do not repeatedly tune the model against these test results.

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Evaluate on the separate test dataset
X_test = test[enhanced_features]
y_test = test["label"]

test_pred = rf.predict(X_test)

print("OFFICIAL TEST RESULTS")
print(confusion_matrix(y_test, test_pred))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=["Normal", "Attack"],
    zero_division=0
))

## 11. Recorded findings and limitations

Results below were obtained in the original Kaggle notebook; **rerunning the code recomputes the metrics**.

| Evaluation | Attack precision | Attack recall | Notes |
|---|---:|---:|---|
| Isolation Forest — 5 features (validation) | 0.697 | 0.054 | 5% normal-derived threshold |
| Isolation Forest — 10 features (validation) | 0.937 | 0.379 | Generic detection dominates |
| Random Forest (validation) | ≈0.93 | 0.9897 | Non-Generic attack recall 0.985 |
| Random Forest (official test) | ≈0.769 | ≈0.9899 | Accuracy ≈0.830 |

**Subgroup finding:** enhanced Isolation Forest detected 91.4% of Generic attacks, but only **11.6% of non-Generic** attacks on the validation subset.

**Test confusion matrix** (`[[TN, FP], [FN, TP]]`): `[[23480, 13520], [456, 44876]]`.
The test false-alarm rate was **13,520 / 37,000 = 36.54%**, versus **16.54%** in validation. The attack-detection rate remained approximately 99%, but the high false-alarm rate limits practical usefulness.

**Caveats:** We compared models repeatedly on one development split; the official test set should not be used for further tuning. Precision depends on the dataset's attack prevalence. Anomaly detection and supervised attack recognition have different learning objectives; these results do not demonstrate detection of entirely novel real-world attacks.